In [1]:
from pathlib import Path
import pandas as pd
import re

# ========= 改成你的路徑 =========
input_csv = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/financial_tables/ethnicity_financial_shares.csv")
output_tex = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/financial_tables/ethnicity_financial_shares.tex")
# ===============================

def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)

# 讀入 CSV
df = pd.read_csv(input_csv)

# 需要欄位
cols = [
    "ethn_group",
    "p_findiff",
    "p_finworse",
    "n",
]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少欄位: {missing}")

df = df[cols].copy()

# 數值欄位轉 numeric
num_cols = ["p_findiff", "p_finworse", "n"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 指定排序
ethnicity_order = [
    "British/English/Scottish/Welsh/Northern Irish",
    "Indian",
    "Pakistani",
    "Bangladeshi",
    "African",
    "Caribbean",
    "Any other white background",
]
df["ethn_group"] = pd.Categorical(
    df["ethn_group"],
    categories=ethnicity_order,
    ordered=True
)
df = df.sort_values("ethn_group").reset_index(drop=True)

# LaTeX escape
df["ethn_group"] = df["ethn_group"].astype(str).map(latex_escape)

# 輸出 LaTeX
lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Financial Difficulties by Ethnicity}")
lines.append(r"\label{tab:ethnicity_financial}")
lines.append(r"\begin{threeparttable}")
lines.append(r"\footnotesize")
lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.8cm}rrr}")
lines.append(r"\toprule")
lines.append(r"Ethnicity & Baseline difficulties & Worse financial status & N \\")
lines.append(r"\midrule")

for _, row in df.iterrows():
    eth = row["ethn_group"]
    fd = "" if pd.isna(row["p_findiff"]) else f'{row["p_findiff"]:.2f}'
    fw = "" if pd.isna(row["p_finworse"]) else f'{row["p_finworse"]:.2f}'
    n = "" if pd.isna(row["n"]) else f'{int(round(row["n"])):,}'
    lines.append(f"{eth} & {fd} & {fw} & {n} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular*}")
lines.append(r"\begin{tablenotes}[flushleft]")
lines.append(r"\footnotesize")
lines.append(
    r"\item Notes: This table reports weighted percentages of baseline financial difficulties and worsening financial status by ethnicity. Baseline financial difficulties are defined as \(fin\_diff\_base = 1\), and worsening financial status is defined as \(fin\_change = 3\). Percentages are weighted using the CA Covid survey weights, and \(N\) denotes the unweighted sample size."
)
lines.append(r"\end{tablenotes}")
lines.append(r"\end{threeparttable}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)

# 寫出 .tex
output_tex.write_text(latex_table, encoding="utf-8")

print(f"LaTeX table saved to: {output_tex}")
print()
print(latex_table)

LaTeX table saved to: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/financial_tables/ethnicity_financial_shares.tex

\begin{table}[htbp]
\centering
\caption{Financial Difficulties by Ethnicity}
\label{tab:ethnicity_financial}
\begin{threeparttable}
\footnotesize
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.8cm}rrr}
\toprule
Ethnicity & Baseline difficulties & Worse financial status & N \\
\midrule
British/English/Scottish/Welsh/Northern Irish & 26.60 & 15.96 & 12,558 \\
Indian & 33.57 & 21.81 & 433 \\
Pakistani & 26.18 & 23.71 & 262 \\
Bangladeshi & 47.52 & 24.03 & 100 \\
African & 66.07 & 14.54 & 112 \\
Caribbean & 62.56 & 17.77 & 131 \\
Any other white background & 31.35 & 19.46 & 417 \\
\bottomrule
\end{tabular*}
\begin{tablenotes}[flushleft]
\footnotesize
\item Notes: This table reports weighted percentages of baseline financial difficulties and worsening financial status by ethnicity. Baseline financial difficulties are defined as \(fin\_diff\_b